Cleaning Data With Pandas



In [ ]:
import pandas as pd

In [ ]:
from google.colab import files

print("Upload the AQI file first:")
uploaded = files.upload()
aqi_df = pd.read_csv('annual_aqi_by_county_2026.csv')

print("Now upload the PLACES asthma file:")
uploaded = files.upload()
asthma_df = pd.read_csv('PLACES__County_Data__GIS_Friendly_Format___2025_release.csv')

Upload the AQI file first:


Saving annual_aqi_by_county_2026.csv to annual_aqi_by_county_2026 (1).csv
Now upload the PLACES asthma file:


Saving PLACES__County_Data__GIS_Friendly_Format___2025_release.csv to PLACES__County_Data__GIS_Friendly_Format___2025_release.csv


In [ ]:
print("=== AQI dataset ===")
print(aqi_df.head())
print(aqi_df.info())

print("\n=== Asthma dataset ===")
print(asthma_df.head())
print(asthma_df.info())

=== AQI dataset ===
     State     County  Year  Days with AQI  Good Days  Moderate Days  \
0  Alabama    Baldwin  2026             85         68             17   
1  Alabama       Clay  2026             87         56             31   
2  Alabama     DeKalb  2026             90         61             29   
3  Alabama     Etowah  2026             90         54             36   
4  Alabama  Jefferson  2026             91         40             51   

   Unhealthy for Sensitive Groups Days  Unhealthy Days  Very Unhealthy Days  \
0                                    0               0                    0   
1                                    0               0                    0   
2                                    0               0                    0   
3                                    0               0                    0   
4                                    0               0                    0   

   Hazardous Days  Max AQI  90th Percentile AQI  Median AQI  Days CO  \


In [ ]:
# Check for missing values in both datasets
print("Missing values in AQI dataset:\n", aqi_df.isnull().sum())
print("\nMissing values in Asthma dataset:\n", asthma_df[['CASTHMA_CrudePrev', 'CASTHMA_AdjPrev']].isnull().sum())

# From exploring this dataset, 187 counties have no asthma value —
# these are small/rural counties where the survey sample was too thin.
# Rather than dropping them (which would silently shrink the dataset),
# flag them so the gap stays visible in later analysis:
asthma_df['has_asthma_data'] = asthma_df['CASTHMA_CrudePrev'].notnull()

Missing values in AQI dataset:
 State                                  0
County                                 0
Year                                   0
Days with AQI                          0
Good Days                              0
Moderate Days                          0
Unhealthy for Sensitive Groups Days    0
Unhealthy Days                         0
Very Unhealthy Days                    0
Hazardous Days                         0
Max AQI                                0
90th Percentile AQI                    0
Median AQI                             0
Days CO                                0
Days NO2                               0
Days Ozone                             0
Days PM2.5                             0
Days PM10                              0
dtype: int64

Missing values in Asthma dataset:
 CASTHMA_CrudePrev    187
CASTHMA_AdjPrev      187
dtype: int64


In [ ]:
# Check for duplicate State+County combinations
dupes = asthma_df[asthma_df.duplicated(subset=['StateAbbr', 'CountyName'], keep=False)]
print("Duplicate-looking county entries:\n", dupes[['StateAbbr', 'CountyName', 'CountyFIPS', 'TotalPopulation']])

# These aren't actual duplicates to drop — they're independent cities
# (e.g., Baltimore city vs. Baltimore county) that share a name.
# Use CountyFIPS, not just State+CountyName, as the real unique key going forward.

Duplicate-looking county entries:
      StateAbbr CountyName  CountyFIPS  TotalPopulation
452         VA    Roanoke       51161            97026
620         VA    Fairfax       51600            25144
877         MO  St. Louis       29189           987059
1078        VA    Roanoke       51770            97171
1340        MO  St. Louis       29510           281754
1579        VA    Fairfax       51059          1141878
1806        VA   Richmond       51760           229247
1825        VA   Richmond       51159             9184
2137        VA   Franklin       51620             8339
2344        MD  Baltimore       24510           565239
2599        VA   Franklin       51067            55549
2898        MD  Baltimore       24005           844703


In [ ]:
# Only convert if it's still a string/object column
if asthma_df['TotalPopulation'].dtype == 'object':
    asthma_df['TotalPopulation'] = asthma_df['TotalPopulation'].str.replace(',', '').astype(int)

print(asthma_df['TotalPopulation'].dtype)  # should show int64

ci_split = asthma_df['CASTHMA_Crude95CI'].str.strip('()').str.split(',', expand=True)
asthma_df['ci_lower'] = pd.to_numeric(ci_split[0], errors='coerce')
asthma_df['ci_upper'] = pd.to_numeric(ci_split[1], errors='coerce')

int64


In [ ]:
merged_df = pd.merge(
    aqi_df,
    asthma_df,
    left_on=['State', 'County'],
    right_on=['StateDesc', 'CountyName'],
    how='outer',
    indicator=True
)

print("Merged rows:", len(merged_df))
print(merged_df['_merge'].value_counts())  # shows how many matched vs. didn't

Merged rows: 3171
_merge
right_only    2341
both           802
left_only       28
Name: count, dtype: int64


Preparing Data with Numpy

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# If continuing in the same notebook session, merged_df is already in memory.
# If starting fresh, re-upload the merged/cleaned file:
# from google.colab import files
# uploaded = files.upload()
# merged_df = pd.read_csv('your_merged_file.csv')

print(merged_df.shape)

(3171, 189)


In [ ]:
# Convert asthma prevalence column to a NumPy array
asthma_rates = merged_df['CASTHMA_CrudePrev'].values

# Replace missing values with the column's mean
mean_asthma = np.nanmean(asthma_rates)
asthma_rates = np.where(np.isnan(asthma_rates), mean_asthma, asthma_rates)

# Assign back to the DataFrame
merged_df['CASTHMA_CrudePrev_filled'] = asthma_rates

print("Mean asthma prevalence used to fill gaps:", round(mean_asthma, 2))

Mean asthma prevalence used to fill gaps: 10.61


In [ ]:
# Asthma prevalence should realistically fall between 0% and 100%
valid_asthma = asthma_rates[(asthma_rates >= 0) & (asthma_rates <= 100)]
merged_df = merged_df[merged_df['CASTHMA_CrudePrev_filled'].isin(valid_asthma)]

# Median AQI shouldn't be negative either — filter any impossible values
merged_df = merged_df[merged_df['Median AQI'] >= 0]

print("Rows remaining after outlier filtering:", len(merged_df))

Rows remaining after outlier filtering: 830


In [ ]:
aqi_values = merged_df['Median AQI'].values

avg_aqi = np.mean(aqi_values)
max_aqi = np.max(aqi_values)
avg_asthma = np.mean(merged_df['CASTHMA_CrudePrev_filled'].values)

print(f"Average Median AQI across counties: {avg_aqi:.2f}")
print(f"Highest Median AQI recorded: {max_aqi}")
print(f"Average asthma prevalence: {avg_asthma:.2f}%")

Average Median AQI across counties: 39.33
Highest Median AQI recorded: 72.0
Average asthma prevalence: 10.60%


In [ ]:
# Normalize Median AQI to a 0-1 scale so it's comparable to asthma prevalence,
# even though the two are measured on very different scales (AQI index vs. %)
min_aqi, max_aqi_val = np.min(aqi_values), np.max(aqi_values)
merged_df['Normalized_AQI'] = (aqi_values - min_aqi) / (max_aqi_val - min_aqi)

min_asthma, max_asthma = np.min(merged_df['CASTHMA_CrudePrev_filled'].values), np.max(merged_df['CASTHMA_CrudePrev_filled'].values)
merged_df['Normalized_Asthma'] = (merged_df['CASTHMA_CrudePrev_filled'] - min_asthma) / (max_asthma - min_asthma)

In [ ]:
merged_df = merged_df.dropna(subset=['TotalPopulation'])
merged_df['TotalPopulation'] = merged_df['TotalPopulation'].astype(np.int64)

In [ ]:
merged_df.to_csv('cleaned_aqi_asthma_data.csv', index=False)

from google.colab import files
files.download('cleaned_aqi_asthma_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Final structural review before moving to analysis

# 1. Re-check for any remaining missing values
print("Remaining missing values:\n", merged_df.isnull().sum())

# 2. Confirm data types are correct
print("\nData types:\n", merged_df.dtypes)

# 3. Confirm no leftover duplicate rows
print("\nDuplicate rows:", merged_df.duplicated().sum())

# 4. Confirm the final row count and how many counties actually matched both datasets
print("\nFinal row count:", len(merged_df))
print(merged_df['_merge'].value_counts())

# 5. Spot-check a few rows visually
print(merged_df.sample(5))

Remaining missing values:
 State                        0
County                       0
Year                         0
Days with AQI                0
Good Days                    0
                            ..
ci_upper                    34
_merge                       0
CASTHMA_CrudePrev_filled     0
Normalized_AQI               0
Normalized_Asthma            0
Length: 192, dtype: int64

Data types:
 State                         object
County                        object
Year                         float64
Days with AQI                float64
Good Days                    float64
                              ...   
ci_upper                     float64
_merge                      category
CASTHMA_CrudePrev_filled     float64
Normalized_AQI               float64
Normalized_Asthma            float64
Length: 192, dtype: object

Duplicate rows: 0

Final row count: 802
_merge
both          802
left_only       0
right_only      0
Name: count, dtype: int64
           State      County  

Normalize and Transform Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Check the actual scale/skew of each column first, to confirm this choice
print(merged_df[['CASTHMA_CrudePrev', 'Median AQI', 'TotalPopulation']].describe())

       CASTHMA_CrudePrev  Median AQI  TotalPopulation
count         768.000000  802.000000     8.020000e+02
mean           10.598568   39.482544     2.934252e+05
std             0.915354    9.450327     5.978938e+05
min             8.000000    2.000000     1.034000e+03
25%            10.000000   37.000000     3.998950e+04
50%            10.600000   40.000000     1.137220e+05
75%            11.200000   44.000000     2.983150e+05
max            14.300000   72.000000     9.663345e+06


In [ ]:
scaler = MinMaxScaler()
merged_df[['CASTHMA_CrudePrev_scaled', 'Median_AQI_scaled']] = scaler.fit_transform(
    merged_df[['CASTHMA_CrudePrev', 'Median AQI']]
)

In [ ]:
# log1p is used instead of log() because it safely handles any value of 0
# (log(0) is undefined, but log1p(0) = 0)
merged_df['TotalPopulation_log'] = np.log1p(merged_df['TotalPopulation'])

In [ ]:
print(merged_df[['CASTHMA_CrudePrev', 'CASTHMA_CrudePrev_scaled',
                  'Median AQI', 'Median_AQI_scaled',
                  'TotalPopulation', 'TotalPopulation_log']].head())

    CASTHMA_CrudePrev  CASTHMA_CrudePrev_scaled  Median AQI  \
1                 9.3                  0.206349        39.0   
13               10.4                  0.380952        40.0   
24               10.1                  0.333333        43.0   
27               10.3                  0.365079        47.0   
36                9.9                  0.301587        52.0   

    Median_AQI_scaled  TotalPopulation  TotalPopulation_log  
1            0.528571           253507            12.443151  
13           0.542857            14111             9.554781  
24           0.585714            72569            11.192307  
27           0.642857           103241            11.544831  
36           0.714286           662895            13.404373  


In [ ]:
summary_table_full = merged_df[[
    'CASTHMA_CrudePrev', 'CASTHMA_CrudePrev_scaled',
    'Median AQI', 'Median_AQI_scaled',
    'TotalPopulation', 'TotalPopulation_log'
]].describe()

print(summary_table_full)

       CASTHMA_CrudePrev  CASTHMA_CrudePrev_scaled  Median AQI  \
count         768.000000                768.000000  802.000000   
mean           10.598568                  0.412471   39.482544   
std             0.915354                  0.145294    9.450327   
min             8.000000                  0.000000    2.000000   
25%            10.000000                  0.317460   37.000000   
50%            10.600000                  0.412698   40.000000   
75%            11.200000                  0.507937   44.000000   
max            14.300000                  1.000000   72.000000   

       Median_AQI_scaled  TotalPopulation  TotalPopulation_log  
count         802.000000     8.020000e+02           802.000000  
mean            0.535465     2.934252e+05            11.620173  
std             0.135005     5.978938e+05             1.431547  
min             0.000000     1.034000e+03             6.942157  
25%             0.500000     3.998950e+04            10.596388  
50%            

In [ ]:
summary_table_full.to_csv('summary_statistics.csv')

from google.colab import files
files.download('summary_statistics.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Perform Data Quality Checks

In [ ]:
def data_quality_report(df):
    print("🔍 Missing Values:\n", df.isnull().sum(), "\n")
    print("📊 Data Types:\n", df.dtypes, "\n")
    print("🗂 Duplicate Rows:", df.duplicated().sum(), "\n")
    print("📈 Summary Stats:\n", df[['CASTHMA_CrudePrev', 'Median AQI', 'TotalPopulation']].describe(), "\n")

    for col in ['StateAbbr', 'StateDesc', 'Dominant Pollutant']:
        if col in df.columns:
            print(f"🔢 Unique values in {col}:\n", df[col].unique(), "\n")

# Run the function
data_quality_report(merged_df)

🔍 Missing Values:
 State                        0
County                       0
Year                         0
Days with AQI                0
Good Days                    0
                            ..
Normalized_AQI               0
Normalized_Asthma            0
CASTHMA_CrudePrev_scaled    34
Median_AQI_scaled            0
TotalPopulation_log          0
Length: 195, dtype: int64 

📊 Data Types:
 State                        object
County                       object
Year                        float64
Days with AQI               float64
Good Days                   float64
                             ...   
Normalized_AQI              float64
Normalized_Asthma           float64
CASTHMA_CrudePrev_scaled    float64
Median_AQI_scaled           float64
TotalPopulation_log         float64
Length: 195, dtype: object 

🗂 Duplicate Rows: 0 

📈 Summary Stats:
        CASTHMA_CrudePrev  Median AQI  TotalPopulation
count         768.000000  802.000000     8.020000e+02
mean           10.598568

In [ ]:
# Duplicate county names (the Baltimore/St. Louis/Roanoke-style cases)
dupe_names = merged_df[merged_df.duplicated(subset=['StateAbbr', 'CountyName'], keep=False)]
print("Counties sharing a name within their state:", len(dupe_names))

# Outlier check on asthma prevalence (should be realistically 0-100%)
outliers = merged_df[(merged_df['CASTHMA_CrudePrev'] < 0) | (merged_df['CASTHMA_CrudePrev'] > 100)]
print("Asthma prevalence outliers:", len(outliers))

# Confirm merge success rate
print(merged_df['_merge'].value_counts())

Counties sharing a name within their state: 6
Asthma prevalence outliers: 0
_merge
both          802
left_only       0
right_only      0
Name: count, dtype: int64
